# Errors, Exceptions, Logging and Debugging

Notebook for **Chapter 6** of the *python* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/python/ch06.html

Tip: change a number, predict the output, then run the cell again.

## 6.1 Reading Tracebacks and the Common Exceptions

In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
def parse_price(text):
    return float(text.replace("Rs", ""))

def total(prices):
    s = 0
    for p in prices:
        s += parse_price(p)
    return s

print(total(["Rs100", "Rs250", "N/A"]))

Traceback (most recent call last):
  File "/private/tmp/w6/tb.py", line 10, in <module>
    print(total(["Rs100", "Rs250", "N/A"]))
          ~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/private/tmp/w6/tb.py", line 7, in total
    s += parse_price(p)
         ~~~~~~~~~~~^^^
  File "/private/tmp/w6/tb.py", line 2, in parse_price
    return float(text.replace("Rs", ""))
ValueError: could not convert string to float: 'N/A'


In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
def attempt(f):
    try:
        print("ok:", f())
    except Exception as e:
        print(type(e).__name__, "->", e)

attempt(lambda: 1/0)
attempt(lambda: int("abc"))
attempt(lambda: [1,2,3][5])
attempt(lambda: {"a":1}["b"])
attempt(lambda: None.upper())
attempt(lambda: "5"+5)
attempt(lambda: undefined_name)
attempt(lambda: open("/no/such/file.txt"))
attempt(lambda: __import__("not_a_module"))
attempt(lambda: (1,2).append(3))

ZeroDivisionError -> division by zero
ValueError -> invalid literal for int() with base 10: 'abc'
IndexError -> list index out of range
KeyError -> 'b'
AttributeError -> 'NoneType' object has no attribute 'upper'
TypeError -> can only concatenate str (not "int") to str
NameError -> name 'undefined_name' is not defined
FileNotFoundError -> [Errno 2] No such file or directory: '/no/such/file.txt'
ModuleNotFoundError -> No module named 'not_a_module'
AttributeError -> 'tuple' object has no attribute 'append'


## 6.2 Handling Errors:try,except,else,finally

In [ ]:
def read_num(s):
    try:
        n = int(s)
    except ValueError:
        print("  bad number")
        return None
    else:
        print("  parsed", n)
        return n
    finally:
        print("  finally runs for", repr(s))

read_num("42")
read_num("x")

  parsed 42
  finally runs for '42'
  bad number
  finally runs for 'x'


In [ ]:
try:
    total = prise + 1      # typo: should be price
except:
    total = 0

In [ ]:
for v in ["10", "0", "zz", None]:
    try:
        print(100 // int(v))
    except (ValueError, TypeError) as e:
        print("input problem:", type(e).__name__)
    except ZeroDivisionError:
        print("zero")

10
zero
input problem: ValueError
input problem: TypeError


In [ ]:
d = {"a": 1}
if "b" in d:
    x = d["b"]
else:
    x = 0
try:
    y = d["b"]
except KeyError:
    y = 0
print(x, y)

for n in [3, 5, 7]:
    if n % 2 == 0:
        print("found even"); break
else:
    print("no even")

0 0
no even


## 6.3 Raising, Custom Exceptions and Chaining

In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
class AppError(Exception):
    """Base error for our app."""

class ValidationError(AppError):
    def __init__(self, field, message):
        super().__init__(f"{field}: {message}")
        self.field = field

def validate(rec):
    if rec.get("age", 0) < 0:
        raise ValidationError("age", "must be >= 0")
    return rec

try:
    validate({"age": -3})
except AppError as e:
    print(type(e).__name__, e, e.field)

ValidationError age: must be >= 0 age


In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
import json, traceback

def load(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError as e:
        raise AppError("config is not valid JSON") from e

try:
    load("{bad")
except AppError as e:
    print(e, "| cause:", type(e.__cause__).__name__)
    traceback.print_exc()

config is not valid JSON | cause: JSONDecodeError
Traceback (most recent call last):
  File "/private/tmp/w6/c.py", line 19, in load
    return json.loads(text)
           ~~~~~~~~~~^^^^^^
  File ".../json/__init__.py", line 352, in loads
    return _default_decoder.decode(s)
    ...
json.decoder.JSONDecodeError: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/private/tmp/w6/c.py", line 23, in <module>
    load("{bad")
  File "/private/tmp/w6/c.py", line 21, in load
    raise AppError("config is not valid JSON") from e
AppError: config is not valid JSON


In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
def risky():
    try:
        1/0
    except ZeroDivisionError:
        print("log and re-raise")
        raise

try:
    risky()
except ZeroDivisionError as e:
    print("caught again:", e)

try:
    assert 1 + 1 == 3, "math broke"
except AssertionError as e:
    print("AssertionError:", e)

try:
    raise ExceptionGroup("batch", [ValueError("a"), TypeError("b")])
except* ValueError as eg:
    print("VE", eg.exceptions)
except* TypeError as eg:
    print("TE", eg.exceptions)

try:
    try:
        int("x")
    except ValueError as e:
        e.add_note("while reading row 7")
        raise
except ValueError as e:
    print(e.__notes__)

log and re-raise
caught again: division by zero
AssertionError: math broke
VE (ValueError('a'),)
TE (TypeError('b'),)
['while reading row 7']


## 6.4 Logging

In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
import logging, sys
logging.basicConfig(stream=sys.stdout, level=logging.INFO,
    format="%(levelname)s | %(name)s | %(message)s")
log = logging.getLogger("etl.load")
log.debug("hidden at INFO level")
log.info("loaded %d rows from %s", 120, "orders.csv")
log.warning("missing column %r", "city")
log.error("cannot connect")
try:
    1/0
except ZeroDivisionError:
    log.exception("calc failed")
print(logging.getLevelName(30), logging.WARNING, logging.ERROR, logging.CRITICAL)

INFO | etl.load | loaded 120 rows from orders.csv
WARNING | etl.load | missing column 'city'
ERROR | etl.load | cannot connect
ERROR | etl.load | calc failed
Traceback (most recent call last):
  File "/private/tmp/w6/lg.py", line 10, in <module>
    1/0
    ~^~
ZeroDivisionError: division by zero
WARNING 30 40 50


In [ ]:
import logging, sys, tempfile, os
d = tempfile.mkdtemp(); fn = os.path.join(d, "app.log")
log = logging.getLogger("svc")
log.setLevel(logging.DEBUG)
fh = logging.FileHandler(fn, encoding="utf-8"); fh.setLevel(logging.DEBUG)
ch = logging.StreamHandler(sys.stdout); ch.setLevel(logging.WARNING)
fmt = logging.Formatter("%(levelname)s:%(name)s:%(message)s")
fh.setFormatter(fmt); ch.setFormatter(fmt)
log.addHandler(fh); log.addHandler(ch)
log.debug("detail for file only")
log.info("info for file only")
log.warning("warning to both")
print("--- file content ---")
fh.flush(); print(open(fn).read(), end="")
child = logging.getLogger("svc.db")
child.error("child logger propagates to parent handlers")

--- file content ---
DEBUG:svc:detail for file only
INFO:svc:info for file only
ERROR:svc.db:child logger propagates to parent handlers


In [ ]:
import logging, sys, json
logging.basicConfig(stream=sys.stdout, level=logging.INFO, format="%(message)s")
class JsonFormatter(logging.Formatter):
    def format(self, r):
        return json.dumps({"level": r.levelname, "msg": r.getMessage(), "logger": r.name})
h = logging.StreamHandler(sys.stdout); h.setFormatter(JsonFormatter())
l2 = logging.getLogger("json"); l2.addHandler(h); l2.propagate = False; l2.setLevel(logging.INFO)
l2.info("order %s paid", 17)

{"level": "INFO", "msg": "order 17 paid", "logger": "json"}


## 6.5 Debugging withpdb

In [ ]:
def average(nums):
    total = 0
    for n in nums:
        total += n
    breakpoint()
    return total / len(nums)

print(average([10, 20, 30]))
print(average([]))

In [ ]:
def buggy_max(nums):
    best = 0
    for n in nums:
        if n > best:
            best = n
    return best
print(buggy_max([-5, -2, -9]))

## 6.6 Production Logging: Structured Logs, dictConfig, Warnings and Crash Tools

Section 6.4 taught the basics of `logging`. In production, nobody reads logs line by line. Programs read them: a log system searches, counts and filters. That needs logs with **structure** (fields, not just sentences) and a **correlation ID** so you can follow one request through many log lines.

In [ ]:
import contextvars, json, logging, logging.config
request_id = contextvars.ContextVar("request_id", default="-")
class JsonFormatter(logging.Formatter):
    def format(self, record):
        payload = {"level": record.levelname, "logger": record.name,
                   "msg": record.getMessage(), "request_id": request_id.get()}
        payload.update(getattr(record, "fields", {}))
        return json.dumps(payload, sort_keys=True)
config = {
    "version": 1,
    "formatters": {"json": {"()": JsonFormatter}},
    "handlers": {"console": {"class": "logging.StreamHandler", "stream": "ext://sys.stdout",
                             "formatter": "json", "level": "INFO"}},
    "loggers": {"shop": {"handlers": ["console"], "level": "DEBUG", "propagate": False}},
}
logging.config.dictConfig(config)
log = logging.getLogger("shop.orders")
def handle(rid, amount):
    request_id.set(rid)
    log.debug("not shown, handler level is INFO")
    log.info("order received", extra={"fields": {"amount": amount}})
    log.error("charge failed", extra={"fields": {"amount": amount}})
handle("req-17", 40)
handle("req-18", 5)

{"amount": 40, "level": "INFO", "logger": "shop.orders", "msg": "order received", "request_id": "req-17"}
{"amount": 40, "level": "ERROR", "logger": "shop.orders", "msg": "charge failed", "request_id": "req-17"}
{"amount": 5, "level": "INFO", "logger": "shop.orders", "msg": "order received", "request_id": "req-18"}
{"amount": 5, "level": "ERROR", "logger": "shop.orders", "msg": "charge failed", "request_id": "req-18"}


In [ ]:
import sys, warnings, faulthandler
def old_api():
    warnings.warn("old_api is deprecated, use new_api", DeprecationWarning, stacklevel=2)
    return 1
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    old_api()
print(caught[0].category.__name__, str(caught[0].message))
with warnings.catch_warnings():
    warnings.simplefilter("error")
    try:
        old_api()
    except DeprecationWarning as e:
        print("promoted to error:", e)
print(faulthandler.is_enabled())
faulthandler.enable()
print(faulthandler.is_enabled())
def crash():
    return {}["missing"]
try:
    crash()
except KeyError:
    tb = sys.exc_info()[2]
    while tb.tb_next:
        tb = tb.tb_next
    print(tb.tb_frame.f_code.co_name)

DeprecationWarning old_api is deprecated, use new_api
promoted to error: old_api is deprecated, use new_api
False
True
crash


## Chapter recap

## Practice set

Problem 6.P.1 What exception does `int("3.5")` raise?

In [ ]:
try:
    print("A")
except ValueError:
    print("B")
else:
    print("C")
finally:
    print("D")

In [ ]:
def f():
    try:
        return "try"
    finally:
        print("cleanup")
print(f())

In [ ]:
try:
    value = int(text)
except:
    pass
print(value * 2)

In [ ]:
def to_float(s):
    try:
        return float(s)
    except ValueError:
        return None

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/python/ch06.html